# MedScribe Agent - MedGemma 4B Real Inference (Kaggle T4)

This notebook tests real MedGemma 4B inference with 4-bit quantization on a Kaggle T4 GPU.

## Prerequisites
- **Kaggle GPU**: Settings → Accelerator → GPU T4 x2
- **HuggingFace Account**: Accept MedGemma license at https://huggingface.co/google/medgemma-4b-it
- **HF Token**: Add as Kaggle Secret named `HF_TOKEN` (Settings → Secrets)

## 1. Install Dependencies

In [ ]:
%%time
# Install core dependencies
!pip install -q transformers>=4.50.0 bitsandbytes>=0.45.0 accelerate>=0.34.0
!pip install -q scispacy pyyaml
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# Install HealthChain from GitHub
!pip install -q healthchain

print("\nDependencies installed.")

## 2. Check GPU & Login to HuggingFace

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")

In [ ]:
from huggingface_hub import login
import os

# On Kaggle, use Secrets to set HF_TOKEN
# On Colab/local, set HF_TOKEN env var or run `huggingface-cli login`
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("Loaded HF token from environment")
    else:
        print("No HF token found. Run: huggingface-cli login")
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")

## 3. Load MedGemma 4B (4-bit Quantized)

In [ ]:
%%time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Model loaded on {model.device}")

# Check VRAM usage
vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"VRAM usage: {vram_used:.1f} GB / {vram_total:.1f} GB ({vram_used/vram_total*100:.0f}%)")

## 4. Test Raw Inference

Basic sanity check: send a simple medical question and verify the model responds coherently.

In [ ]:
%%time

messages = [
    {"role": "user", "content": "What is the ICD-10 code for community-acquired pneumonia?"}
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

outputs = model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.3)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Prompt: What is the ICD-10 code for community-acquired pneumonia?")
print(f"\nResponse:\n{response}")

## 5. Test Clinical Reasoning Prompt

This tests the actual reasoning prompt template used by MedScribeAgent, and validates that the model returns parseable structured JSON.

In [ ]:
import json
import re
import time

SAMPLE_NOTE = """65-year-old male admitted for community-acquired pneumonia.
History of type 2 diabetes on metformin. Chest X-ray shows right lower lobe infiltrate.
Started on azithromycin IV.
Assessment: Community-acquired pneumonia, type 2 diabetes."""

REASONING_PROMPT = """You are a medical coder. Given the clinical note, extract ONLY the confirmed medical diagnoses from the Assessment line. Do NOT code symptoms, lab values, demographics, medications, or procedures as diagnoses.

Rules:
- Only code conditions explicitly listed in the Assessment/Diagnosis section
- Each diagnosis needs an ICD-10 code
- If the note has no clear diagnoses, return an empty diagnoses array
- Keep the list short: only confirmed conditions

Respond with a single JSON object, no markdown fences, no explanation:
{"diagnoses": [{"text": "...", "icd10": "...", "confidence": "high", "gaps": ""}], "procedures": [{"text": "...", "cpt": "..."}], "medications": [{"text": "...", "status": "active"}], "discrepancies": []}
"""

messages = [
    {"role": "system", "content": REASONING_PROMPT},
    {"role": "user", "content": f"Clinical Note:\n{SAMPLE_NOTE}"},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

start = time.time()
outputs = model.generate(inputs, max_new_tokens=512, do_sample=True, temperature=0.3)
latency = time.time() - start

raw_response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print(f"Inference latency: {latency:.1f}s")
print(f"Output tokens: {outputs.shape[-1] - inputs.shape[-1]}")
print(f"\nRaw response:\n{raw_response}")

In [ ]:
def parse_json_response(response, fallback):
    """Extract JSON from model response, handling markdown fences and arrays."""
    cleaned = response.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list):
            return {"diagnoses": parsed, "procedures": [], "medications": [], "discrepancies": []}
    except (json.JSONDecodeError, TypeError):
        pass

    json_match = re.search(r"\{[\s\S]*\}", cleaned)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    array_match = re.search(r"\[[\s\S]*\]", cleaned)
    if array_match:
        try:
            parsed = json.loads(array_match.group())
            if isinstance(parsed, list):
                return {"diagnoses": parsed, "procedures": [], "medications": [], "discrepancies": []}
        except json.JSONDecodeError:
            pass

    print("WARNING: Could not parse JSON from response")
    return fallback

parsed = parse_json_response(
    raw_response,
    {"diagnoses": [], "procedures": [], "medications": [], "discrepancies": []}
)

print("Parsed JSON:")
print(json.dumps(parsed, indent=2))

print(f"\n--- Validation ---")
print(f"Diagnoses: {len(parsed.get('diagnoses', []))}")
print(f"Procedures: {len(parsed.get('procedures', []))}")
print(f"Medications: {len(parsed.get('medications', []))}")

for dx in parsed.get("diagnoses", []):
    print(f"  {dx.get('text', 'N/A'):40s} ICD-10: {dx.get('icd10', 'N/A'):10s} [{dx.get('confidence', 'N/A')}]")

## 6. Test Resolution Prompt

Test the discrepancy resolution prompt with simulated NLP vs LLM discrepancies.

In [ ]:
RESOLUTION_PROMPT = """You are a medical coder resolving discrepancies between NLP extraction and prior analysis.

Rules:
- Only resolve items that are actual medical diagnoses or conditions
- Ignore demographics (age, gender), lab values, medication names, and general terms
- If a discrepancy item is not a real diagnosis, exclude it from the resolved list
- Keep the resolved list short: only confirmed conditions with ICD-10 codes

Respond with a single JSON object, no markdown fences, no explanation:
{"resolved": [{"text": "...", "code": "...", "reasoning": "...", "confidence": "high"}]}
"""

discrepancies = [
    {"type": "nlp_only", "entity": "right lower lobe infiltrate", "reason": "Found by NLP but not confirmed by LLM"},
    {"type": "llm_only", "entity": "uncontrolled diabetes", "icd10": "E11.65", "reason": "LLM inferred but NLP did not extract"}
]

user_content = (
    f"Original Clinical Note:\n{SAMPLE_NOTE}\n\n"
    f"Discrepancies to resolve:\n{json.dumps(discrepancies, indent=2)}\n\n"
    f"Relevant Patient History (FHIR):\n"
    f"Observation: HbA1c 7.8% (2 days ago)\n"
    f"Condition: Type 2 diabetes mellitus (active, diagnosed 2019)"
)

messages = [
    {"role": "system", "content": RESOLUTION_PROMPT},
    {"role": "user", "content": user_content},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

start = time.time()
outputs = model.generate(inputs, max_new_tokens=512, do_sample=True, temperature=0.3)
latency = time.time() - start

raw = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Resolution latency: {latency:.1f}s")
print(f"\nRaw response:\n{raw}")

resolution = parse_json_response(raw, {"resolved": []})
print(f"\nParsed resolution:")
print(json.dumps(resolution, indent=2))

## 7. Test via MedGemmaClient Class

Now test through the actual `MedGemmaClient` class to verify the full code path works.

In [ ]:
import sys
import os

# Add project root to path (adjust for your setup)
# On Kaggle: clone the repo first, then add to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Force real mode since we have a GPU
os.environ["MEDGEMMA_MODE"] = "real"
os.environ["MEDGEMMA_MODEL"] = "google/medgemma-4b-it"

import logging
logging.basicConfig(level=logging.INFO)

from medscribe.src.models.medgemma import create_client

client = create_client()
print(f"Client type: {type(client).__name__}")

In [ ]:
%%time

# Test reason_over_note through the client
result = client.reason_over_note(SAMPLE_NOTE)
print("reason_over_note result:")
print(json.dumps(result, indent=2))

In [ ]:
%%time

# Test reason_with_resolution through the client
discrepancies = [
    {"type": "nlp_only", "entity": "right lower lobe infiltrate", "reason": "Found by NLP but not confirmed by LLM"}
]
resolution = client.reason_with_resolution(SAMPLE_NOTE, discrepancies, patient_history="Condition: Type 2 DM (active)")
print("reason_with_resolution result:")
print(json.dumps(resolution, indent=2))

## 8. End-to-End Pipeline with Real MedGemma

Run the full MedScribeAgent pipeline: NLP extraction → real MedGemma reasoning → validation → output.

In [ ]:
%%time
from healthchain.io import Document
from medscribe.src.pipeline.coding_pipeline import build_coding_pipeline_simple
from medscribe.src.agent.orchestrator import MedScribeAgent

# Build components
pipeline = build_coding_pipeline_simple()

agent = MedScribeAgent(
    coding_pipeline=pipeline,
    medgemma_client=client,
    fhir_gateway=None,
)

# Run full agent pipeline
output = agent.run(SAMPLE_NOTE, patient_id="Patient/demo-001")

print(f"\n{'='*60}")
print("AGENT RUN COMPLETE (REAL MEDGEMMA)")
print(f"{'='*60}")

In [ ]:
# Inspect results
print(f"NLP Entities: {len(output['nlp_entities'])}")
for e in output['nlp_entities']:
    print(f"  {e.get('text', 'N/A'):30s} [{e.get('label', 'N/A')}]")

print(f"\nLLM Diagnoses: {len(output['llm_reasoning'].get('diagnoses', []))}")
for dx in output['llm_reasoning'].get('diagnoses', []):
    print(f"  {dx.get('text', 'N/A'):40s} ICD-10: {dx.get('icd10', 'N/A'):10s} [{dx.get('confidence', 'N/A')}]")

print(f"\nDiscrepancies: {len(output['discrepancies'])}")
for d in output['discrepancies']:
    print(f"  [{d['type']}] {d.get('entity', 'N/A')}: {d.get('reason', '')}")

print(f"\nCDS Cards: {len(output['cds_cards'])}")
for card in output['cds_cards']:
    print(f"  [{card.indicator.value:8s}] {card.summary}")

## 9. Latency Benchmark

Run multiple notes and measure average latency.

In [ ]:
TEST_NOTES = [
    SAMPLE_NOTE,
    """72-year-old female with worsening dyspnea and edema. Echo shows EF 35%.
    EKG shows atrial fibrillation. Started metoprolol and furosemide.
    Assessment: Congestive heart failure, atrial fibrillation.""",
    """78-year-old female in ICU with sepsis from urinary tract infection.
    Urine culture positive for E. coli. Creatinine elevated to 3.2 from baseline 0.9
    indicating acute kidney injury. Started piperacillin-tazobactam.
    Assessment: Sepsis, UTI, acute kidney injury.""",
]

latencies = []
for i, note in enumerate(TEST_NOTES):
    start = time.time()
    result = client.reason_over_note(note)
    elapsed = time.time() - start
    latencies.append(elapsed)
    n_dx = len(result.get("diagnoses", []))
    print(f"Note {i+1}: {elapsed:.1f}s | {n_dx} diagnoses")

print(f"\nAverage latency: {sum(latencies)/len(latencies):.1f}s")
print(f"Min: {min(latencies):.1f}s | Max: {max(latencies):.1f}s")

## 10. Summary

Record results for the project writeup.

In [ ]:
print("MedScribe Agent - MedGemma 4B Results")
print("=" * 50)
print(f"Model: {MODEL_ID}")
print(f"Quantization: 4-bit NF4 (bitsandbytes)")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"Avg inference latency: {sum(latencies)/len(latencies):.1f}s")
print(f"End-to-end pipeline: {'PASS' if output.get('cds_cards') else 'FAIL'}")
print()
print("Benchmark Results (6 test cases):")
print(f"  Mean F1:        {mean_f1:.3f}")
print(f"  Mean Precision: {mean_prec:.3f}")
print(f"  Mean Recall:    {mean_rec:.3f}")
print(f"  Mean Latency:   {mean_lat:.1f}s/case")

## 11. Benchmark Evaluation (F1, Precision, Recall)

Run the full agent pipeline on 6 test cases across multiple clinical specialties and compute ICD-10 coding accuracy.

In [ ]:
TEST_CASES = [
    {
        "note_id": "TC-001",
        "scenario": "Pneumonia + diabetes (inpatient)",
        "patient_id": "Patient/test-001",
        "clinical_note": "65-year-old male admitted for community-acquired pneumonia. History of type 2 diabetes on metformin. Chest X-ray shows right lower lobe infiltrate. Started on azithromycin IV. Assessment: Community-acquired pneumonia, type 2 diabetes.",
        "ground_truth_codes": ["J18.9", "E11.9"],
    },
    {
        "note_id": "TC-002",
        "scenario": "Heart failure + atrial fibrillation",
        "patient_id": "Patient/test-002",
        "clinical_note": "72-year-old female with worsening dyspnea and edema. Echo shows EF 35%. EKG shows atrial fibrillation. Started metoprolol and furosemide. Assessment: Congestive heart failure, atrial fibrillation.",
        "ground_truth_codes": ["I50.9", "I48.91"],
    },
    {
        "note_id": "TC-003",
        "scenario": "Sepsis + AKI + UTI (critical care)",
        "patient_id": "Patient/test-003",
        "clinical_note": "78-year-old female in ICU with sepsis from urinary tract infection. Urine culture positive for E. coli. Creatinine elevated to 3.2 from baseline 0.9 indicating acute kidney injury. Started piperacillin-tazobactam. Assessment: Sepsis, UTI, acute kidney injury.",
        "ground_truth_codes": ["A41.9", "N39.0", "N17.9"],
    },
    {
        "note_id": "TC-004",
        "scenario": "Depression + anxiety (behavioral health)",
        "patient_id": "Patient/test-004",
        "clinical_note": "45-year-old female follow-up for depression and anxiety. PHQ-9 score 16. GAD-7 score 14. On sertraline, increased to 100mg daily. Assessment: Major depression, generalized anxiety disorder.",
        "ground_truth_codes": ["F32.9", "F41.9"],
    },
    {
        "note_id": "TC-005",
        "scenario": "Pneumonia only (single condition)",
        "patient_id": "Patient/test-005",
        "clinical_note": "55-year-old male with 5 days of cough and fever. Chest X-ray confirms right lower lobe pneumonia. No significant past history. Started amoxicillin. Assessment: Community-acquired pneumonia.",
        "ground_truth_codes": ["J18.9"],
    },
    {
        "note_id": "TC-006",
        "scenario": "Vague note (edge case)",
        "patient_id": "Patient/test-006",
        "clinical_note": "Patient seen today. Reports feeling unwell for a week. No specific complaints. Vital signs stable. Exam unremarkable. Labs pending. Follow up as needed.",
        "ground_truth_codes": [],
    },
]

print(f"Loaded {len(TEST_CASES)} test cases")

In [ ]:
%%time
# Run each test case through the full agent pipeline

def calculate_f1(predicted, ground_truth):
    predicted, ground_truth = set(predicted), set(ground_truth)
    if not predicted and not ground_truth:
        return 1.0, 1.0, 1.0
    if not predicted or not ground_truth:
        return 0.0, 0.0, 0.0
    tp = predicted & ground_truth
    prec = len(tp) / len(predicted)
    rec = len(tp) / len(ground_truth)
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return f1, prec, rec

results = []
for case in TEST_CASES:
    start = time.time()
    output = agent.run(case["clinical_note"], case["patient_id"])
    elapsed = time.time() - start

    predicted = [dx.get("icd10", "") for dx in output.get("llm_reasoning", {}).get("diagnoses", [])]
    gt = case["ground_truth_codes"]
    f1, prec, rec = calculate_f1(predicted, gt)

    results.append({
        "note_id": case["note_id"],
        "scenario": case["scenario"],
        "f1": f1, "precision": prec, "recall": rec,
        "latency": elapsed,
        "predicted": predicted, "gt": gt,
    })
    print(f"{case['note_id']}: F1={f1:.3f} Prec={prec:.3f} Rec={rec:.3f} Lat={elapsed:.1f}s | Predicted: {predicted}")

print(f"\n{'='*70}")
mean_f1 = sum(r["f1"] for r in results) / len(results)
mean_prec = sum(r["precision"] for r in results) / len(results)
mean_rec = sum(r["recall"] for r in results) / len(results)
mean_lat = sum(r["latency"] for r in results) / len(results)
print(f"MEAN  F1={mean_f1:.3f}  Precision={mean_prec:.3f}  Recall={mean_rec:.3f}  Latency={mean_lat:.1f}s")
print(f"{'='*70}")